# 🏠 Hyderabad Housing Price Prediction Model

## Interactive Machine Learning Tutorial

This notebook will:
1. Load your housing dataset
2. Explore and visualize the data
3. Preprocess features
4. Train a Random Forest model
5. Evaluate performance
6. Save the trained model

---

## 📦 Step 1: Install Required Libraries

Run this cell once to install all dependencies.

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn joblib openpyxl -q

## 🔧 Step 2: Import Libraries

Import all necessary Python libraries for data analysis and modeling.

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Save model
import joblib

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Set plot style
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 📂 Step 3: Load Dataset

Load your CSV file. Make sure `housing_data.csv` is in the same folder as this notebook.

In [ ]:
# Load the dataset
df = pd.read_csv('housing_data.csv')

print('=' * 70)
print('🏠 SEATTLE HOUSING DATASET LOADED SUCCESSFULLY!')
print('=' * 70)
print(f'\n📊 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\n💾 Memory Usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')
print('\n' + '=' * 70)

## 🔍 Step 4: Explore the Data

Let's take a look at what our data contains!

In [ ]:
# Display first 5 rows
print('📋 FIRST 5 ROWS OF DATA:')
df.head()

In [ ]:
# Dataset info
print('📊 DATASET INFORMATION:')
print('=' * 70)
df.info()
print('\n' + '=' * 70)

In [ ]:
# Statistical summary
print('📈 STATISTICAL SUMMARY:')
print('=' * 70)
df.describe().round(2)

In [ ]:
# Check for missing values
print('🔍 MISSING VALUES CHECK:')
print('=' * 70)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage (%)': missing_pct
}).sort_values('Missing Count', ascending=False)

print(missing_df[missing_df['Missing Count'] > 0])
if missing.sum() == 0:
    print('\n✅ No missing values found!')
print('\n' + '=' * 70)

## 📊 Step 5: Data Visualization

Visualize price distribution and key relationships.

In [ ]:
# Price distribution
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.histplot(df['price'], bins=50, kde=True)
plt.title('Price Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Price (USD)')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
sns.boxplot(y=df['price'])
plt.title('Price Box Plot (Outliers)', fontsize=14, fontweight='bold')
plt.ylabel('Price (USD)')

plt.tight_layout()
plt.show()

print(f"\n💰 Price Statistics:")
print(f"   Min: ${df['price'].min():,.0f}")
print(f"   Max: ${df['price'].max():,.0f}")
print(f"   Mean: ${df['price'].mean():,.0f}")
print(f"   Median: ${df['price'].median():,.0f}")

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## ⚙️ Step 6: Data Preprocessing

Prepare the data for machine learning by handling missing values and encoding categorical variables.

In [ ]:
# Handle missing values (if any)
print('🔧 HANDLING MISSING VALUES...')
print('=' * 70)

# Fill numeric missing values with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'✅ Filled missing {col} with median: {median_val}')

# Drop rows with missing categorical values (if any)
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().any():
        df = df.dropna(subset=[col])
        print(f'✅ Dropped rows with missing {col}')

print(f'\nFinal dataset size: {len(df):,} rows')
print('\n' + '=' * 70)

In [ ]:
# Encode categorical variables
print('🔢 ENCODING CATEGORICAL VARIABLES...')
print('=' * 70)

# Create encoders
encoders = {}

# Encode city
if 'city' in df.columns:
    le_city = LabelEncoder()
    df['city_encoded'] = le_city.fit_transform(df['city'].astype(str))
    encoders['city'] = le_city
    print(f'✅ Encoded {len(le_city.classes_)} unique cities')

# Extract and encode ZIP code
if 'statezip' in df.columns:
    df['zipcode'] = df['statezip'].astype(str).str[-5:]
    le_zipcode = LabelEncoder()
    df['zipcode_encoded'] = le_zipcode.fit_transform(df['zipcode'])
    encoders['zipcode'] = le_zipcode
    print(f'✅ Encoded {len(le_zipcode.classes_)} unique ZIP codes')

# Convert date column
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df['year_sold'] = df['date'].dt.year
    df['month_sold'] = df['date'].dt.month
    print('✅ Extracted year and month from date')

print('\n' + '=' * 70)

## 🎯 Step 7: Feature Selection

Select the features we'll use for prediction.

In [ ]:
# Define feature columns
exclude_cols = ['price', 'date', 'street', 'country', 'statezip', 'city']
if 'zipcode' in df.columns:
    exclude_cols.append('zipcode')

feature_cols = [col for col in df.columns if col not in exclude_cols]

print('🎯 SELECTED FEATURES FOR PREDICTION:')
print('=' * 70)
for i, col in enumerate(feature_cols, 1):
    print(f'{i:2d}. {col}')

print(f'\nTotal features: {len(feature_cols)}')
print('\n' + '=' * 70)

In [ ]:
# Prepare X and y
X = df[feature_cols]
y = df['price']

print(f'\n📊 Feature Matrix Shape: {X.shape}')
print(f'📊 Target Vector Shape: {y.shape}')
print(f'\n✅ Features ready for training!')

## 🔄 Step 8: Train-Test Split & Scaling

Split data into training and testing sets, then scale features.

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 80% train, 20% test
    random_state=42,    # For reproducibility
    shuffle=True        # Shuffle before splitting
)

print('🔄 TRAIN-TEST SPLIT:')
print('=' * 70)
print(f'Training samples: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)')
print(f'Testing samples:  {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)')
print('\n' + '=' * 70)

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('⚖️  FEATURE SCALING COMPLETE')
print('=' * 70)
print('✅ StandardScaler applied to all features')
print('   Mean after scaling: ~0')
print('   Std after scaling: ~1')
print('\n' + '=' * 70)

## 🌲 Step 9: Train Random Forest Model

Now we train the machine learning model!

In [ ]:
# Create and train the model
print('🌲 TRAINING RANDOM FOREST MODEL...')
print('=' * 70)

rf_model = RandomForestRegressor(
    n_estimators=200,       # Number of trees
    max_depth=20,           # Maximum depth
    min_samples_split=5,    # Minimum samples to split
    min_samples_leaf=2,     # Minimum samples per leaf
    random_state=42,
    n_jobs=-1              # Use all CPU cores
)

print('\n⏳ Training in progress... (this may take 1-2 minutes)')
print('Please wait...\n')

rf_model.fit(X_train_scaled, y_train)

print('✅ TRAINING COMPLETE!')
print('\n' + '=' * 70)

## 📊 Step 10: Model Evaluation

Evaluate how well our model performs on test data.

In [ ]:
# Make predictions
y_pred_train = rf_model.predict(X_train_scaled)
y_pred_test = rf_model.predict(X_test_scaled)

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2 = r2_score(y_test, y_pred_test)

train_mae = mean_absolute_error(y_train, y_pred_train)
train_r2 = r2_score(y_train, y_pred_train)

print('📊 MODEL EVALUATION RESULTS')
print('=' * 70)
print('\n🎯 TEST SET PERFORMANCE:')
print(f'   MAE (Mean Absolute Error):   ${mae:,.0f}')
print(f'   RMSE (Root Mean Squared Error): ${rmse:,.0f}')
print(f'   R² Score:                    {r2:.4f} ({r2*100:.2f}%)')

print('\n🎯 TRAINING SET PERFORMANCE:')
print(f'   MAE: ${train_mae:,.0f}')
print(f'   R²:  {train_r2:.4f} ({train_r2*100:.2f}%)')

# Check for overfitting
print('\n' + '-' * 70)
if abs(r2 - train_r2) > 0.1:
    print('⚠️  WARNING: Possible overfitting detected!')
    print(f'   Gap between train ({train_r2:.2f}) and test ({r2:.2f}) R² is large')
else:
    print('✅ Model generalization looks good!')
    print('   Train and test performance are consistent')

print('\n' + '=' * 70)

## 🔍 Step 11: Feature Importance Analysis

See which features are most important for predicting prices.

In [ ]:
# Calculate feature importance
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('🔍 FEATURE IMPORTANCE ANALYSIS')
print('=' * 70)
print('\nTop 10 Most Important Features:\n')
print(feature_importance.head(10).to_string(index=False))
print('\n' + '=' * 70)

# Visualize feature importance
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['Importance'].values)
plt.yticks(range(len(top_features)), top_features['Feature'].values)
plt.xlabel('Importance', fontsize=12)
plt.title('Top 15 Most Important Features', fontsize=16, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 🧪 Step 12: Test Sample Predictions

Try the model on a few sample properties to see how it performs.

In [ ]:
# Test on 5 sample properties
sample_indices = X_test.index[:5]
sample_properties = X_test.loc[sample_indices]
sample_actual_prices = y_test.loc[sample_indices]

# Scale and predict
sample_scaled = scaler.transform(sample_properties)
sample_predictions = rf_model.predict(sample_scaled)

print('🧪 SAMPLE PROPERTY PREDICTIONS')
print('=' * 80)

for i, idx in enumerate(sample_indices):
    actual = sample_actual_prices.loc[idx]
    predicted = sample_predictions[i]
    error = abs(predicted - actual)
    error_pct = (error / actual) * 100
    
    print(f'\n🏠 Property #{i+1}:')
    print(f'   Actual Price:    ${actual:,.0f}')
    print(f'   Predicted Price: ${predicted:,.0f}')
    print(f'   Difference:      ${error:,.0f} ({error_pct:.1f}%)')
    print(f'   Accuracy:        {100 - error_pct:.1f}%')

print('\n' + '=' * 80)

## 💾 Step 13: Save the Model

Save all trained components so we can use them later without retraining.

In [ ]:
print('💾 SAVING MODEL AND ARTIFACTS...')
print('=' * 70)

# Save model
joblib.dump(rf_model, 'hyderabad_housing_model.pkl')
print('✅ Model saved to: hyderabad_housing_model.pkl')

# Save scaler
joblib.dump(scaler, 'scaler.pkl')
print('✅ Scaler saved to: scaler.pkl')

# Save encoders
if 'city' in df.columns:
    joblib.dump(encoders['city'], 'city_encoder.pkl')
    print('✅ City encoder saved to: city_encoder.pkl')

if 'statezip' in df.columns:
    joblib.dump(encoders['zipcode'], 'zipcode_encoder.pkl')
    print('✅ ZIP code encoder saved to: zipcode_encoder.pkl')

# Save feature list
joblib.dump(feature_cols, 'feature_columns.pkl')
print('✅ Feature columns saved to: feature_columns.pkl')

print('\n🎉 ALL FILES SAVED SUCCESSFULLY!')
print('\n' + '=' * 70)

## 🎯 Summary & Next Steps

### ✅ What We Accomplished:
- Loaded and explored the Hyderabad housing dataset
- Preprocessed data (handled missing values, encoded categories)
- Trained a Random Forest model with 200 trees
- Achieved **R² = {r2*100:.2f}%** accuracy
- Saved the complete model pipeline

### 📊 Final Performance:
- **Test MAE**: ${mae:,.0f}
- **Test RMSE**: ${rmse:,.0f}
- **Test R²**: {r2:.4f} ({r2*100:.2f}%)

### 🚀 Next Steps:
1. ✅ Model is saved and ready to use!
2. 📝 Run `predict_price.py` to test custom predictions
3. 🔌 Follow INTEGRATION_GUIDE.md to add to your Node.js app
4. 📈 Deploy to production and start making predictions!

### 💡 How to Use This Model:

```python
import joblib

# Load the trained model
model = joblib.load('hyderabad_housing_model.pkl')
scaler = joblib.load('scaler.pkl')
features = joblib.load('feature_columns.pkl')

# Prepare your property data
property_data = {
    'bedrooms': 3,
    'bathrooms': 2.5,
    'sqft_living': 2000,
    # ... add all features
}

# Make prediction
import pandas as pd
df = pd.DataFrame([property_data])
X = df[features]
X_scaled = scaler.transform(X)
predicted_price = model.predict(X_scaled)[0]

print(f'Predicted Price: ${predicted_price:,.2f}')
```

---

**🎉 CONGRATULATIONS! Your ML model is ready for production!** 🚀